In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
cifar_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()  # Converts to tensor and normalizes to [0, 1]
)

print(f"CIFAR-10 loaded: {len(cifar_data)} images")
print(f"Image shape: {cifar_data[0][0].shape}")  # Should be (3, 32, 32)

In [ ]:
class ColorizationDataset(Dataset):
    def __init__(self, cifar10_dataset):
        self.cifar10_dataset = cifar10_dataset

    def __len__(self):
        """
        Returns the total number of images in the dataset.
        """
        return len(self.cifar10_dataset)

    def __getitem__(self, idx):
        color_image_tensor, _ = self.cifar10_dataset[idx]  # Already a tensor from transforms.ToTensor()

        # Extract R, G, B channels
        R = color_image_tensor[0, :, :]
        G = color_image_tensor[1, :, :]
        B = color_image_tensor[2, :, :]

        # Convert to grayscale using the formula
        grayscale_image_tensor = 0.299 * R + 0.587 * G + 0.114 * B

        # Add channel dimension to make it (1, H, W)
        grayscale_image_tensor = grayscale_image_tensor.unsqueeze(0)

        return grayscale_image_tensor, color_image_tensor

In [ ]:
# Test your implementation
colorization_dataset = ColorizationDataset(cifar_data)

# Get a sample
gray_img, color_img = colorization_dataset[0]

print(f"Grayscale image shape: {gray_img.shape}")  # Should be (1, 32, 32)
print(f"Color image shape: {color_img.shape}")      # Should be (3, 32, 32)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(gray_img.squeeze(), cmap='gray')
axes[0].set_title('Grayscale (Input)')
axes[0].axis('off')
axes[1].imshow(color_img.permute(1, 2, 0))
axes[1].set_title('Color (Target)')
axes[1].axis('off')
plt.show()

In [ ]:
# Test with DataLoader
dataloader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)

gray_batch, color_batch = next(iter(dataloader))
print(f"Batch grayscale shape: {gray_batch.shape}")  # Should be (8, 1, 32, 32)
print(f"Batch color shape: {color_batch.shape}")      # Should be (8, 3, 32, 32)